In [1]:
# load_cases (dataframe with historical + LHS_MDA cases)

import xarray as xr

cases = xr.open_dataset(
    "/lustre/geocean/WORK/users/Jared/HyCoFfEE/02_LHS_MDA/ALL_CASES_Santona.nc"
).isel(index=slice(0, 10))
cases

<xarray.Dataset> Size: 880B
Dimensions:   (index: 10)
Coordinates:
  * index     (index) int64 80B 0 1 2 3 4 5 6 7 8 9
Data variables:
    Qp        (index) float64 80B ...
    Qb        (index) float64 80B ...
    Tp        (index) float64 80B ...
    Tau_ss_d  (index) float64 80B ...
    Sp        (index) float64 80B ...
    Sb        (index) float64 80B ...
    CM        (index) float64 80B ...
    Tau       (index) float64 80B ...
    wind      (index) float64 80B ...
    wind_dir  (index) float64 80B ...

In [2]:
import os.path as op
import pandas as pd

# Calculate the hydrogram parameters for the case.
p_hidro = "inputs/"
hidrograma = "UH_SCS.xlsx"
hidro = pd.read_excel(op.join(p_hidro, hidrograma))
name_columns = hidro.columns
new_names = ["t", "Q"]
hidro.columns = new_names
new_data = {"t": 0, "Q": 0}
new_df = pd.DataFrame([new_data])
hidro = pd.concat([new_df, hidro]).reset_index(drop=True)
hidro.head()

,t,Q
0,0.0,0.00
1,0.1,0.03
2,0.2,0.10
3,0.3,0.19
4,0.4,0.31


In [3]:
fixed_parameters = {
    "start_year": 2000,
    "start_month": 2,
    "start_day": 1,
    "start_hour": 0,
    "utc_start": 8,
    "dt": 5,  # dt = 120 -> timestem (s) # nspool = 360; by default. It saves a data each nspool * dt = 30*120 = 3600 s = 1 h,
    "nspool": 360,
    "ihfskip": 518400,
    "nws": 1,  # wind form .th
    "ics": 2,  # Coordinate option = 1: Cartesian; 2: lon/lat
    "hidro": hidro,
}

In [4]:
metamodel_parameters = {
    "Qp": cases.Qp.values.tolist(),  # List of Qp values for each case (m^3/s)
    "Qb": cases.Qb.values.tolist(),  # List of Qb values for each case (m^3/s)
    "Sp": cases.Sp.values.tolist(),  # List of Sp values for each case (m)
    "Sb": cases.Sb.values.tolist(),  # List of Sb values for each case (m)
    "CM": cases.CM.values.tolist(),  # List of CM values for each case (m)
    "Tp": cases.Tp.values.tolist(),  # List of Tp values for each case (s)
    "Tau_ss_d": cases.Tau_ss_d.values.tolist(),  # List of Th values for each case (s)
    "Tau": cases.Tau.values.tolist(),  # List of Tau values for each case (s)
    "wind_magnitude": cases.wind.values.tolist(),  # Wind magnitudes (m/s) for each case
    "wind_direction": cases.wind_dir.values.tolist(),  # Wind directions (degrees) for each case
}

In [5]:
from bluemath_tk.wrappers.schism.schism_wrapper import SchismHyCoFfEE

schism_wrapper = SchismHyCoFfEE(
    templates_dir="templates",
    metamodel_parameters=metamodel_parameters,
    fixed_parameters=fixed_parameters,
    output_dir="CASES_HyCoFfEE",
)
schism_wrapper

2025-07-10 12:52:59,361 - SchismHyCoFfEE - WARNING - Parameter Qp is not in the default_parameters
2025-07-10 12:52:59,362 - SchismHyCoFfEE - WARNING - Parameter Qb is not in the default_parameters
2025-07-10 12:52:59,362 - SchismHyCoFfEE - WARNING - Parameter Sp is not in the default_parameters
2025-07-10 12:52:59,363 - SchismHyCoFfEE - WARNING - Parameter Sb is not in the default_parameters
2025-07-10 12:52:59,363 - SchismHyCoFfEE - WARNING - Parameter CM is not in the default_parameters
2025-07-10 12:52:59,363 - SchismHyCoFfEE - WARNING - Parameter Tp is not in the default_parameters
2025-07-10 12:52:59,364 - SchismHyCoFfEE - WARNING - Parameter Tau_ss_d is not in the default_parameters
2025-07-10 12:52:59,364 - SchismHyCoFfEE - WARNING - Parameter Tau is not in the default_parameters
2025-07-10 12:52:59,364 - SchismHyCoFfEE - WARNING - Parameter wind_magnitude is not in the default_parameters
2025-07-10 12:52:59,365 - SchismHyCoFfEE - WARNING - Parameter wind_direction is not in th

In [6]:
schism_wrapper.build_cases(mode="one_by_one")

In [7]:
# schism_wrapper.run_cases(launcher="geoocean-cluster", num_workers=10)

In [8]:
schism_wrapper.run_cases_bulk(launcher="sbatch --array=1-10 sbatch_example.sh")

Submitted batch job 645483
